# Real Big Benchmark: GAP + MILP + Genetic

Ноутбук запускает 3 real-метода на **полном** датасете `dataset_real_spb_ready.json`.
Если рядом есть `dataset_real_spb_af_1.json`, приоритет остаётся за ним (как в старом ноутбуке).


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import importlib
import inspect
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for mod_name in list(sys.modules):
    if mod_name.startswith("flowopt"):
        sys.modules.pop(mod_name, None)

import flowopt.pipeline as fp
fp = importlib.reload(fp)

DATA_ROOT = fp.DATA_ROOT
run_real_gap_vrp = fp.run_real_gap_vrp
run_real_gap_vrp_alns = fp.run_real_gap_vrp_alns
run_real_milp = fp.run_real_milp
run_real_genetic = fp.run_real_genetic

milp_sig = inspect.signature(run_real_milp)
if "time_limit_sec" not in milp_sig.parameters:
    raise RuntimeError(f"Stale run_real_milp loaded: signature={milp_sig}")

pd.set_option("display.max_colwidth", 160)
print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("pipeline module:", fp.__file__)
print("run_real_gap_vrp_alns signature:", inspect.signature(run_real_gap_vrp_alns))


REPO_ROOT: /Users/igoreshka/Desktop/Optimization-of-flows
DATA_ROOT: /Users/igoreshka/Desktop/Optimization-of-flows/src/data
pipeline module: /Users/igoreshka/Desktop/Optimization-of-flows/src/flowopt/pipeline.py
run_real_milp signature: (*, dataset_path: 'Path | str' = PosixPath('/Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/real_simple/dataset_real_spb_simple.json'), time_limit_sec: 'int' = 60, unassigned_penalty: 'float' = 100000.0, show_progress: 'bool' = False, progress_hook: 'Callable[[str], None] | None' = None) -> 'RunMetrics'


In [6]:
import json

NOTEBOOK_DIR = DATA_ROOT.parent
DATA_DIR = NOTEBOOK_DIR / "data" / "real"

BASE_DATASET_PATH = DATA_DIR / "dataset_real_spb_ready.json"
WORK_DATASET_PATH = DATA_DIR / "dataset_real_spb_af_1.json"
SIMPLE_DATASET_PATH = DATA_DIR / "real_simple" / "dataset_real_spb_simple.json"
SUMMARY_PATH = DATA_DIR / "summary_real_spb_ready.json"

if not BASE_DATASET_PATH.exists():
    raise FileNotFoundError(f"BASE_DATASET_PATH not found: {BASE_DATASET_PATH}")

# Этот ноутбук специально под большой датасет
if WORK_DATASET_PATH.exists():
    DATASET_PATH = WORK_DATASET_PATH
else:
    DATASET_PATH = BASE_DATASET_PATH

print("BASE_DATASET_PATH:", BASE_DATASET_PATH)
print("WORK_DATASET_PATH:", WORK_DATASET_PATH)
print("SIMPLE_DATASET_PATH:", SIMPLE_DATASET_PATH)
print("DATASET_PATH     :", DATASET_PATH)
print("exists           :", DATASET_PATH.exists())

summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8")) if SUMMARY_PATH.exists() else {}
summary


BASE_DATASET_PATH: /Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/dataset_real_spb_ready.json
WORK_DATASET_PATH: /Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/dataset_real_spb_af_1.json
SIMPLE_DATASET_PATH: /Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/real_simple/dataset_real_spb_simple.json
DATASET_PATH     : /Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/dataset_real_spb_ready.json
exists           : True


{'dataset_name': 'real_data_notebook_pack_march_2026_af_1',
 'counts': {'road_nodes': 46730,
  'road_edges': 109901,
  'mno': 216,
  'object1': 9,
  'depots': 4,
  'agents': 626,
  'tasks': 220,
  'routes': 0},
 'preprocess': {'density_kg_m3': 76.7,
  'tasks_before_sampling': 29518,
  'tasks_after_sampling': 220,
  'agent_fraction': 1.0,
  'active_transport_rows_before_fraction': 786,
  'active_transport_rows_before_agent_sampling': 626,
  'active_transport_rows': 626,
  'objects': 9,
  'zone_object_edges': 26,
  'total_task_mass_t': 72.25216700000001,
  'total_object_capacity_t': 5424.657534246576,
  'container_to_vehicle_types': {'A': ['VT_A', 'VT_AB', 'VT_ABD', 'VT_AD'],
   'B': ['VT_AB', 'VT_ABD'],
   'C': ['VT_C', 'VT_CD'],
   'D': ['VT_ABD', 'VT_AD', 'VT_CD']},
  'vehicle_profiles': {'VT_A': {'max_daily_km': 280.0,
    'max_shift_hours': 14.0,
    'avg_speed_kmph': 26.0,
    'capacity_median_t': 3.92},
   'VT_AB': {'max_daily_km': 320.0,
    'max_shift_hours': 14.0,
    'avg_spee

In [ ]:
# GAP-VRP
GAP_STEP1_METHOD = "lp"
GAP_ITER = 40
MAX_AGENTS = None
USE_REPAIR = True

# ALNS
ALNS_ITERATIONS = 150
ALNS_REMOVAL_Q = 30
ALNS_RANDOM_SEED = 42
ALNS_START_TEMP = 1000
ALNS_END_TEMP = 1
ALNS_STEP = 0.999

# Real MILP
RMILP_TIME_LIMIT_SEC = 60
RMILP_UNASSIGNED_PENALTY = 1e5

# Genetic
GA_POPULATION_SIZE = 40
GA_GENERATIONS = 60
GA_ELITE_SIZE = 4
GA_SEED = 42

# Live progress in notebook output
SHOW_ALGO_PROGRESS = True
SHOW_SOLVER_DETAILS = False

from time import perf_counter


def make_progress_logger(enabled: bool):
    if not enabled:
        return None
    t0 = perf_counter()
    def _log(message: str) -> None:
        dt = perf_counter() - t0
        print(f"[+{dt:7.1f}s] {message}", flush=True)
    return _log


progress_log = make_progress_logger(SHOW_ALGO_PROGRESS)


def run_with_live_status(label, fn, **kwargs):
    if progress_log is not None:
        progress_log(f"{label}: START")
    ts = perf_counter()
    out = fn(**kwargs)
    if progress_log is not None:
        progress_log(f"{label}: DONE in {perf_counter() - ts:.1f}s")
    return out


results = [
    run_with_live_status(
        "GAP-VRP",
        run_real_gap_vrp,
        dataset_path=DATASET_PATH,
        step1_method=GAP_STEP1_METHOD,
        gap_iter=GAP_ITER,
        max_agents=MAX_AGENTS,
        use_repair=USE_REPAIR,
        show_progress=SHOW_SOLVER_DETAILS,
        verbose=False,
        progress_hook=progress_log,
    ),
    run_with_live_status(
        "GAP-VRP-ALNS",
        run_real_gap_vrp_alns,
        dataset_path=DATASET_PATH,
        step1_method=GAP_STEP1_METHOD,
        gap_iter=GAP_ITER,
        max_agents=MAX_AGENTS,
        use_repair=USE_REPAIR,
        alns_iterations=ALNS_ITERATIONS,
        alns_removal_q=ALNS_REMOVAL_Q,
        alns_seed=ALNS_RANDOM_SEED,
        start_temperature=ALNS_START_TEMP,
        end_temperature=ALNS_END_TEMP,
        temperature_step=ALNS_STEP,
        show_progress=SHOW_SOLVER_DETAILS,
        verbose=False,
        progress_hook=progress_log,
    ),
    run_with_live_status(
        "REAL-MILP",
        run_real_milp,
        dataset_path=DATASET_PATH,
        time_limit_sec=RMILP_TIME_LIMIT_SEC,
        unassigned_penalty=RMILP_UNASSIGNED_PENALTY,
        show_progress=SHOW_SOLVER_DETAILS,
        progress_hook=progress_log,
    ),
    run_with_live_status(
        "REAL-GENETIC",
        run_real_genetic,
        dataset_path=DATASET_PATH,
        population_size=GA_POPULATION_SIZE,
        generations=GA_GENERATIONS,
        elite_size=GA_ELITE_SIZE,
        seed=GA_SEED,
        show_progress=SHOW_SOLVER_DETAILS,
        progress_hook=progress_log,
    ),
]

benchmark_df = pd.DataFrame([r.as_dict() for r in results])
benchmark_df = benchmark_df.sort_values(
    by=["all_checks_ok", "transport_work_ton_km", "total_km", "runtime_sec"],
    ascending=[False, True, True, True],
    na_position="last",
).reset_index(drop=True)


# Save benchmark artifact to local JSON
from datetime import datetime

LOCAL_OUT_DIR = REPO_ROOT / "notebooks" / "local" / "real_data"
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULT_JSON_PATH = LOCAL_OUT_DIR / f"benchmark_{RUN_TAG}.json"

records = benchmark_df.where(pd.notna(benchmark_df), None).to_dict(orient="records")
artifact = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "notebook": "real_full_3algo_benchmark_gap-vrp-alns.ipynb",
    "dataset_path": str(DATASET_PATH),
    "config": {
        "gap_step1_method": GAP_STEP1_METHOD,
        "gap_iter": GAP_ITER,
        "max_agents": MAX_AGENTS,
        "use_repair": USE_REPAIR,
        "alns_iterations": ALNS_ITERATIONS,
        "alns_removal_q": ALNS_REMOVAL_Q,
        "alns_seed": ALNS_RANDOM_SEED,
        "alns_start_temp": ALNS_START_TEMP,
        "alns_end_temp": ALNS_END_TEMP,
        "alns_step": ALNS_STEP,
        "rmilp_time_limit_sec": RMILP_TIME_LIMIT_SEC,
        "rmilp_unassigned_penalty": RMILP_UNASSIGNED_PENALTY,
        "ga_population_size": GA_POPULATION_SIZE,
        "ga_generations": GA_GENERATIONS,
        "ga_elite_size": GA_ELITE_SIZE,
        "ga_seed": GA_SEED,
        "show_algo_progress": SHOW_ALGO_PROGRESS,
        "show_solver_details": SHOW_SOLVER_DETAILS,
    },
    "results": records,
}
RESULT_JSON_PATH.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", RESULT_JSON_PATH)

main_cols = [
    "algorithm",
    "feasible",
    "all_checks_ok",
    "assigned_routes",
    "unassigned_tasks",
    "active_agents",
    "transport_work_ton_km",
    "total_km",
    "deadhead_km",
    "deadhead_share_pct",
    "total_hours",
    "runtime_sec",
    "alns_gain",
    "solver_error",
]
benchmark_df[[c for c in main_cols if c in benchmark_df.columns]]


[+    0.0s] GAP-VRP: START
[+    0.0s] [real_gap_vrp] load dataset: /Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/dataset_real_spb_ready.json
[+    1.0s] [real_gap_vrp] dataset loaded (nodes=46730, edges=127100, agents=626, tasks=220)
[+    1.0s] [real_gap_vrp] build nx graph
[+    1.3s] [real_gap_vrp] solver start
[+    1.3s] [real_gap_vrp] [GAP-VRP] start: GAP-Lagrangean + VRP(NN+2opt) [step1=lp]
[+    1.3s] [real_gap_vrp] [GAP-VRP] step1: task generation
[+   68.4s] [real_gap_vrp] [GAP-VRP] step1 done in 67.2s, tasks=216
[+   68.5s] [real_gap_vrp] [GAP-VRP] step2: GAP Lagrangean, iter=40
[+   68.5s] [real_gap_vrp] [GAP-LR] prepare matrices: agents=626, tasks=216, max_iter=40
[+   71.3s] [real_gap_vrp] [GAP-LR] precompute costs: 62/626 agents
[+   71.3s] [real_gap_vrp] [GAP-LR] precompute costs: 124/626 agents
[+   71.8s] [real_gap_vrp] [GAP-LR] precompute costs: 186/626 agents
[+   74.1s] [real_gap_vrp] [GAP-LR] precompute costs: 248/626 agents
[+   78.6s] [real_gap_v

,algorithm,feasible,all_checks_ok,assigned_routes,unassigned_tasks,active_agents,transport_work_ton_km,total_km,deadhead_km,deadhead_share_pct,total_hours,runtime_sec
0,real_gap_vrp,True,True,216,0,60,625.807,5167.712,3218.920,62.289,272.846,721.251
1,real_milp,True,True,220,0,126,683.540,6653.226,4439.617,66.729,339.238,65.535
2,real_genetic,True,True,220,0,197,683.540,8878.035,6664.427,75.066,435.582,487.200


In [ ]:
detail_cols = [
    "algorithm",
    "checks",
    "solver_error",
    "step1_method",
    "gap_iter",
    "used_agents",
    "time_limit_sec",
    "unassigned_penalty",
    "population_size",
    "generations",
    "generations_requested",
    "generation_scale",
    "elite_size",
    "alns_iterations",
    "alns_removal_q",
    "alns_seed",
    "alns_base_objective",
    "alns_best_objective",
    "alns_gain",
]
benchmark_df[[c for c in detail_cols if c in benchmark_df.columns]]


,algorithm,checks,step1_method,gap_iter,used_agents,time_limit_sec,unassigned_penalty,population_size,generations,generations_requested,generation_scale,elite_size
0,real_gap_vrp,"{'hard_constraints_ok': True, 'daily_limits_ok': True, 'reachability_ok': True, 'all_tasks_assigned': True, 'mno_coverage_ok': True, 'all_checks_ok': True}",lp,40.0,626.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,real_milp,"{'hard_constraints_ok': True, 'daily_limits_ok': True, 'reachability_ok': True, 'all_tasks_assigned': True, 'mno_coverage_ok': True, 'all_checks_ok': True}",NaN,NaN,NaN,60.0,100000.0,NaN,NaN,NaN,NaN,NaN
2,real_genetic,"{'hard_constraints_ok': True, 'daily_limits_ok': True, 'reachability_ok': True, 'all_tasks_assigned': True, 'mno_coverage_ok': True, 'all_checks_ok': True}",NaN,NaN,NaN,NaN,NaN,40.0,6.0,60.0,0.1,4.0


In [9]:
from flowopt.reporting import solution_breakdown_tables
from IPython.display import Markdown, display

MAX_AGENTS_TO_SHOW = 30
MAX_TASK_IDS_IN_CELL = 12
MAX_TASK_ROWS_TO_SHOW = 400

for _, row in benchmark_df.iterrows():
    algo = row.get("algorithm", "unknown")
    tables = solution_breakdown_tables(
        row,
        max_agents=MAX_AGENTS_TO_SHOW,
        max_task_ids=MAX_TASK_IDS_IN_CELL,
    )

    display(Markdown(f"### {algo}: решение по агентам"))
    display(tables["summary"])

    if tables["agents"].empty:
        print("No active agents in solution")
        continue

    display(tables["agents"])
    display(Markdown(f"**{algo}: задача -> агент (первые {MAX_TASK_ROWS_TO_SHOW} строк)**"))
    display(tables["tasks"].head(MAX_TASK_ROWS_TO_SHOW))


### real_gap_vrp: решение по агентам

,algorithm,active_agents,assigned_tasks,total_mass_tons,total_km,total_hours
0,real_gap_vrp,60,216,0.0,5167.712,272.846


,agent_id,vehicle_type,is_compact,tasks_count,routes_count,total_mass_tons,total_km,deadhead_km,deadhead_share_pct,total_hours,task_ids
0,AGENT_00276,VT_AD,True,9,9,0,108.643,54.772,50.415,7.818,"LP_T0135, LP_T0023, LP_T0025, LP_T0167, LP_T0007, LP_T0184, LP_T0011, LP_T0015, LP_T0098"
1,AGENT_00278,VT_AD,True,9,9,0,111.045,55.581,50.053,7.927,"LP_T0125, LP_T0107, LP_T0139, LP_T0070, LP_T0010, LP_T0145, LP_T0186, LP_T0113, LP_T0177"
2,AGENT_00277,VT_AD,False,8,8,0,109.235,54.676,50.054,7.525,"LP_T0123, LP_T0102, LP_T0009, LP_T0006, LP_T0187, LP_T0141, LP_T0003, LP_T0012"
3,AGENT_00247,VT_A,True,7,7,0,100.147,60.342,60.253,5.713,"LP_T0143, LP_T0048, LP_T0106, LP_T0109, LP_T0052, LP_T0047, LP_T0066"
4,AGENT_00280,VT_AD,True,7,7,0,108.372,55.174,50.912,7.166,"LP_T0089, LP_T0190, LP_T0004, LP_T0150, LP_T0152, LP_T0005, LP_T0115"
5,AGENT_00279,VT_AD,True,7,7,0,110.798,55.479,50.072,7.276,"LP_T0016, LP_T0085, LP_T0144, LP_T0038, LP_T0185, LP_T0183, LP_T0146"
6,AGENT_00256,VT_AB,False,7,7,0,115.095,65.203,56.651,6.336,"LP_T0133, LP_T0058, LP_T0105, LP_T0002, LP_T0073, LP_T0046, LP_T0130"
7,AGENT_00252,VT_AB,False,7,7,0,125.266,71.105,56.763,6.839,"LP_T0101, LP_T0122, LP_T0181, LP_T0061, LP_T0043, LP_T0021, LP_T0169"
8,AGENT_00250,VT_AB,False,6,6,0,119.867,68.983,57.550,6.434,"LP_T0138, LP_T0079, LP_T0166, LP_T0092, LP_T0067, LP_T0095"
9,AGENT_00254,VT_AB,False,6,6,0,127.036,83.715,65.899,6.613,"LP_T0062, LP_T0110, LP_T0124, LP_T0034, LP_T0171, LP_T0210"


**real_gap_vrp: задача -> агент (первые 400 строк)**

,algorithm,agent_id,task_order,task_id
0,real_gap_vrp,AGENT_00276,1,LP_T0135
1,real_gap_vrp,AGENT_00276,2,LP_T0023
2,real_gap_vrp,AGENT_00276,3,LP_T0025
3,real_gap_vrp,AGENT_00276,4,LP_T0167
4,real_gap_vrp,AGENT_00276,5,LP_T0007
...,...,...,...,...
211,real_gap_vrp,AGENT_00395,1,LP_T0201
212,real_gap_vrp,AGENT_00399,1,LP_T0202
213,real_gap_vrp,AGENT_00391,1,LP_T0206
214,real_gap_vrp,AGENT_00396,1,LP_T0212


...

### real_genetic: решение по агентам

,algorithm,active_agents,assigned_tasks,total_mass_tons,total_km,total_hours
0,real_genetic,197,220,72.243,8878.035,435.582


,agent_id,vehicle_type,is_compact,tasks_count,routes_count,total_mass_tons,total_km,deadhead_km,deadhead_share_pct,total_hours,task_ids
0,AGENT_00252,VT_AB,False,2,2,0.983,14.851,8.203,55.233,1.059,"TASK_00102, TASK_00136"
1,AGENT_00262,VT_AB,False,2,2,0.851,21.516,11.729,54.515,1.356,"TASK_00146, TASK_00085"
2,AGENT_00256,VT_AB,False,2,2,0.168,23.017,16.517,71.760,1.399,"TASK_00154, TASK_00089"
3,AGENT_00277,VT_AD,False,2,2,0.456,24.181,13.080,54.092,1.739,"TASK_00143, TASK_00151"
4,AGENT_00317,VT_AB,False,2,2,0.230,26.329,19.831,75.318,1.557,"TASK_00131, TASK_00107"
5,AGENT_00059,VT_AB,False,2,2,0.515,31.972,17.535,54.846,1.792,"TASK_00070, TASK_00010"
6,AGENT_00268,VT_AB,False,2,2,0.288,32.518,22.117,68.013,1.795,"TASK_00111, TASK_00159"
7,AGENT_00330,VT_AB,False,2,2,0.109,33.003,21.182,64.181,1.815,"TASK_00114, TASK_00140"
8,AGENT_00619,VT_AD,False,2,2,0.506,34.291,25.034,73.002,2.199,"TASK_00112, TASK_00117"
9,AGENT_00622,VT_AD,False,2,2,0.450,34.431,25.095,72.886,2.205,"TASK_00097, TASK_00087"


**real_genetic: задача -> агент (первые 400 строк)**

,algorithm,agent_id,task_order,task_id
0,real_genetic,AGENT_00252,1,TASK_00102
1,real_genetic,AGENT_00252,2,TASK_00136
2,real_genetic,AGENT_00262,1,TASK_00146
3,real_genetic,AGENT_00262,2,TASK_00085
4,real_genetic,AGENT_00256,1,TASK_00154
...,...,...,...,...
215,real_genetic,AGENT_00533,1,TASK_00217
216,real_genetic,AGENT_00454,1,TASK_00199
217,real_genetic,AGENT_00443,1,TASK_00200
218,real_genetic,AGENT_00438,1,TASK_00207
